# In Class Assignment

**Name:** Christine Wu
**Date:** 04/23/26

## Submission Instructions
* Submit a link to your Jupyter Notebook in your GitHub repository. Make sure your settings will allow viewing of your notebook.

* Your notebook must be fully run (all outputs visible) and committed and pushed to your repository before the deadline.



## Activity:

## Imports

In [26]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

from xgboost import XGBClassifier


## Load the data

In [27]:

adult = pd.read_csv('/Users/christinewu/Downloads/adult.csv')
adult.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


## Clean the data and prepare the target

In [28]:
adult = adult.copy()

adult = adult.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

adult["income"] = adult["income"].apply(lambda x: 1 if x == ">50K" else 0)

adult.drop(columns=["fnlwgt"], inplace=True, errors="ignore")

adult["gender"] = adult["gender"].apply(lambda x: 1 if x == "Male" else 0)

adult.replace("?", np.nan, inplace=True)

for col in adult.columns:
    if adult[col].dtype == "object":
        adult[col] = adult[col].fillna("unknown")
    else:
        adult[col] = adult[col].fillna(adult[col].median())

adult.head()


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,unknown,Some-college,10,Never-married,unknown,Own-child,White,0,0,0,30,United-States,0


## Feature preprocessing

In [29]:

df_fe = adult.copy()

X = df_fe.drop("income", axis=1)
y = df_fe["income"]

X_train_fe, X_test_fe, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = X_train_fe.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()

# collapse rare labels using a 1 percent threshold
for col in cat_cols:
    value_freq = X_train_fe[col].value_counts(normalize=True)
    rare_levels = value_freq[value_freq < 0.01].index.tolist()

    X_train_fe[col] = X_train_fe[col].replace(rare_levels, "Rare")
    X_test_fe[col] = X_test_fe[col].where(~X_test_fe[col].isin(rare_levels), "Rare")

# count or frequency encoding
for col in cat_cols:
    freq_map = X_train_fe[col].value_counts(normalize=True).to_dict()
    X_train_fe[col] = X_train_fe[col].map(freq_map)
    X_test_fe[col] = X_test_fe[col].map(freq_map).fillna(0)

# discretize selected numeric features with equal-frequency style bins
disc_vars = [col for col in num_cols if col not in ["gender", "capital-gain", "capital-loss"]]

for col in disc_vars:
    try:
        _, bins = pd.qcut(X_train_fe[col], q=10, duplicates="drop", retbins=True)
        bins[0] = -np.inf
        bins[-1] = np.inf

        X_train_fe[col] = pd.cut(X_train_fe[col], bins=bins, labels=False, include_lowest=True)
        X_test_fe[col] = pd.cut(X_test_fe[col], bins=bins, labels=False, include_lowest=True)
    except ValueError:
        pass

# drop constant features
constant_features = [col for col in X_train_fe.columns if X_train_fe[col].nunique() <= 1]
X_train_fe = X_train_fe.drop(columns=constant_features)
X_test_fe = X_test_fe.drop(columns=constant_features)

print("Dropped constant features:", constant_features)
print("Feature-engine style train shape:", X_train_fe.shape)
X_train_fe.head()


Dropped constant features: []
Feature-engine style train shape: (39073, 13)


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
34342,9,0.693727,0.324751,1,0.330945,0.042664,0.258618,0.85609,1,0,0,0,0.898344
18559,0,0.693727,0.028613,0,0.330945,0.113122,0.031352,0.85609,0,0,0,0,0.898344
12477,2,0.693727,0.324751,1,0.456632,0.101144,0.402580,0.85609,1,0,0,2,0.064699
560,6,0.693727,0.324751,1,0.031684,0.115067,0.105239,0.09513,0,0,0,2,0.898344
3427,3,0.693727,0.164257,4,0.456632,0.125278,0.402580,0.85609,1,0,0,2,0.898344


## Automatic polynomial feature generation

In [30]:

poly = PolynomialFeatures(
    degree=3,
    interaction_only=True,
    include_bias=False
)

X_train_poly = poly.fit_transform(X_train_fe)
X_test_poly = poly.transform(X_test_fe)

poly_feature_names = poly.get_feature_names_out(X_train_fe.columns)

X_train_poly_df = pd.DataFrame(
    X_train_poly,
    columns=poly_feature_names,
    index=X_train_fe.index
)

X_test_poly_df = pd.DataFrame(
    X_test_poly,
    columns=poly_feature_names,
    index=X_test_fe.index
)

print("Selected FE train shape:", X_train_fe.shape)
print("Expanded train shape:", X_train_poly_df.shape)
print("Selected FE test shape:", X_test_fe.shape)
print("Expanded test shape:", X_test_poly_df.shape)


Selected FE train shape: (39073, 13)
Expanded train shape: (39073, 377)
Selected FE test shape: (9769, 13)
Expanded test shape: (9769, 377)


## Baseline models on the 377 engineered features

In [31]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
spw = neg / pos

rf_baseline_poly = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

xgb_baseline_poly = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42
)

rf_cv_scores_poly = cross_val_score(
    rf_baseline_poly,
    X_train_poly_df,
    y_train,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1
)

xgb_cv_scores_poly = cross_val_score(
    xgb_baseline_poly,
    X_train_poly_df,
    y_train,
    cv=skf,
    scoring="balanced_accuracy"
)

rf_baseline_poly.fit(X_train_poly_df, y_train)
xgb_baseline_poly.fit(X_train_poly_df, y_train)

rf_preds_poly = rf_baseline_poly.predict(X_test_poly_df)
xgb_preds_poly = xgb_baseline_poly.predict(X_test_poly_df)

print("Random Forest full-feature CV balanced accuracy:", rf_cv_scores_poly.mean())
print("Random Forest full-feature test balanced accuracy:", balanced_accuracy_score(y_test, rf_preds_poly))
print()
print("XGBoost full-feature CV balanced accuracy:", xgb_cv_scores_poly.mean())
print("XGBoost full-feature test balanced accuracy:", balanced_accuracy_score(y_test, xgb_preds_poly))


Random Forest full-feature CV balanced accuracy: 0.7781034158212777
Random Forest full-feature test balanced accuracy: 0.7910425472372632

XGBoost full-feature CV balanced accuracy: 0.8363291604233176
XGBoost full-feature test balanced accuracy: 0.8363754640784755


## Rank engineered features using the demo notebook approach

In [32]:

# First rank by built-in feature importance
rf_importances_poly = pd.Series(
    rf_baseline_poly.feature_importances_,
    index=X_train_poly_df.columns
).sort_values(ascending=False)

xgb_importances_poly = pd.Series(
    xgb_baseline_poly.feature_importances_,
    index=X_train_poly_df.columns
).sort_values(ascending=False)

print("Top 15 Random Forest feature importances:")
print(rf_importances_poly.head(15))
print()
print("Top 15 XGBoost feature importances:")
print(xgb_importances_poly.head(15))


Top 15 Random Forest feature importances:
marital-status                                   0.043920
marital-status native-country                    0.025701
age marital-status hours-per-week                0.025220
age educational-num marital-status               0.024080
age marital-status occupation                    0.015624
age marital-status                               0.014371
age educational-num occupation                   0.013835
educational-num marital-status hours-per-week    0.013767
age educational-num hours-per-week               0.013216
marital-status hours-per-week                    0.013212
marital-status occupation native-country         0.012701
age educational-num relationship                 0.012644
age marital-status race                          0.012642
marital-status race                              0.012104
age educational-num                              0.010978
dtype: float64

Top 15 XGBoost feature importances:
age educational-num marital-status  

## Permutation importance and average ranking

In [33]:
rf_shortlist = rf_importances_poly.head(40).index.tolist()
xgb_shortlist = xgb_importances_poly.head(40).index.tolist()

rf_short_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_short_model.fit(X_train_poly_df[rf_shortlist], y_train)

xgb_short_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42
)
xgb_short_model.fit(X_train_poly_df[xgb_shortlist], y_train)

perm_idx = X_test_poly_df.sample(n=min(300, len(X_test_poly_df)), random_state=42).index

rf_perm_importance_poly = permutation_importance(
    rf_short_model,
    X_test_poly_df.loc[perm_idx, rf_shortlist],
    y_test.loc[perm_idx],
    n_repeats=3,
    random_state=42,
    scoring="balanced_accuracy",
    n_jobs=-1
)

xgb_perm_importance_poly = permutation_importance(
    xgb_short_model,
    X_test_poly_df.loc[perm_idx, xgb_shortlist],
    y_test.loc[perm_idx],
    n_repeats=3,
    random_state=42,
    scoring="balanced_accuracy"
)

rf_perm_importance_poly_df = pd.DataFrame({
    "feature": rf_shortlist,
    "importance_mean": rf_perm_importance_poly.importances_mean,
    "importance_std": rf_perm_importance_poly.importances_std
}).sort_values(by="importance_mean", ascending=False)

xgb_perm_importance_poly_df = pd.DataFrame({
    "feature": xgb_shortlist,
    "importance_mean": xgb_perm_importance_poly.importances_mean,
    "importance_std": xgb_perm_importance_poly.importances_std
}).sort_values(by="importance_mean", ascending=False)

# Average the model importance rank and permutation importance rank
rf_feat_df = pd.DataFrame({
    "feature": rf_shortlist,
    "feat_importance": rf_importances_poly.loc[rf_shortlist].values
})
rf_feat_df["rank_feat"] = rf_feat_df["feat_importance"].rank(ascending=False)
rf_perm_importance_poly_df["rank_perm"] = rf_perm_importance_poly_df["importance_mean"].rank(ascending=False)

rf_combined = rf_feat_df.merge(
    rf_perm_importance_poly_df[["feature", "importance_mean", "rank_perm"]],
    on="feature"
)
rf_combined["rank_avg"] = (rf_combined["rank_feat"] + rf_combined["rank_perm"]) / 2
rf_combined = rf_combined.sort_values("rank_avg")

xgb_feat_df = pd.DataFrame({
    "feature": xgb_shortlist,
    "feat_importance": xgb_importances_poly.loc[xgb_shortlist].values
})
xgb_feat_df["rank_feat"] = xgb_feat_df["feat_importance"].rank(ascending=False)
xgb_perm_importance_poly_df["rank_perm"] = xgb_perm_importance_poly_df["importance_mean"].rank(ascending=False)

xgb_combined = xgb_feat_df.merge(
    xgb_perm_importance_poly_df[["feature", "importance_mean", "rank_perm"]],
    on="feature"
)
xgb_combined["rank_avg"] = (xgb_combined["rank_feat"] + xgb_combined["rank_perm"]) / 2
xgb_combined = xgb_combined.sort_values("rank_avg")

rf_top_features_20 = rf_combined.head(20)["feature"].tolist()
xgb_top_features_20 = xgb_combined.head(20)["feature"].tolist()

print("Random Forest top 20 reduced features:")
print(rf_top_features_20)
print()
print("XGBoost top 20 reduced features:")
print(xgb_top_features_20)
print()
print("Consistently important features in both top 20 lists:")
print(sorted(set(rf_top_features_20).intersection(set(xgb_top_features_20))))


Random Forest top 20 reduced features:
['educational-num marital-status hours-per-week', 'age educational-num hours-per-week', 'age marital-status occupation', 'age relationship hours-per-week', 'age educational-num relationship', 'educational-num occupation hours-per-week', 'age marital-status', 'marital-status occupation native-country', 'marital-status occupation hours-per-week', 'age educational-num occupation', 'marital-status hours-per-week', 'age marital-status relationship', 'marital-status occupation', 'age educational-num', 'occupation relationship hours-per-week', 'educational-num relationship hours-per-week', 'marital-status race', 'age marital-status hours-per-week', 'marital-status', 'educational-num marital-status']

XGBoost top 20 reduced features:
['marital-status', 'age educational-num hours-per-week', 'age marital-status hours-per-week', 'capital-gain', 'educational-num occupation hours-per-week', 'age educational-num race', 'age educational-num gender', 'age educati

## Create reduced datasets

In [34]:

X_train_rf_20 = X_train_poly_df[rf_top_features_20]
X_test_rf_20 = X_test_poly_df[rf_top_features_20]

X_train_xgb_20 = X_train_poly_df[xgb_top_features_20]
X_test_xgb_20 = X_test_poly_df[xgb_top_features_20]

print("Random Forest reduced train shape:", X_train_rf_20.shape)
print("XGBoost reduced train shape:", X_train_xgb_20.shape)


Random Forest reduced train shape: (39073, 20)
XGBoost reduced train shape: (39073, 20)


## Task 1. Train at least two models that differ in a meaningful way, such as model type or tuning choices, and use your reduced feature sets for these models.

In [35]:

# Train two different model types on the reduced feature sets

rf_model_reduced = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

xgb_model_reduced = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42
)

rf_model_reduced.fit(X_train_rf_20, y_train)
xgb_model_reduced.fit(X_train_xgb_20, y_train)

rf_pred_reduced = rf_model_reduced.predict(X_test_rf_20)
xgb_pred_reduced = xgb_model_reduced.predict(X_test_xgb_20)

rf_reduced_bal_acc = balanced_accuracy_score(y_test, rf_pred_reduced)
xgb_reduced_bal_acc = balanced_accuracy_score(y_test, xgb_pred_reduced)

print("Random Forest reduced-feature test balanced accuracy:", rf_reduced_bal_acc)
print("XGBoost reduced-feature test balanced accuracy:", xgb_reduced_bal_acc)


Random Forest reduced-feature test balanced accuracy: 0.7693564655682004
XGBoost reduced-feature test balanced accuracy: 0.8303484731327471


## Task 2. Tune your models beyond their default settings. There is no fixed number of parameters you must tune, but you should make a reasonable effort to improve performance and demonstrate that your tuning choices had an effect.

In [36]:

# Small grids keep the search reasonable for a laptop while still showing the effect of tuning

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 3]
}

xgb_param_grid = {
    "n_estimators": [80, 120],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_grid=rf_param_grid,
    scoring="balanced_accuracy",
    cv=3,
    n_jobs=-1
)

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=spw,
        eval_metric="logloss",
        random_state=42
    ),
    param_grid=xgb_param_grid,
    scoring="balanced_accuracy",
    cv=3
)

rf_grid.fit(X_train_rf_20, y_train)
xgb_grid.fit(X_train_xgb_20, y_train)

rf_best = rf_grid.best_estimator_
xgb_best = xgb_grid.best_estimator_

rf_tuned_pred = rf_best.predict(X_test_rf_20)
xgb_tuned_pred = xgb_best.predict(X_test_xgb_20)

rf_tuned_bal_acc = balanced_accuracy_score(y_test, rf_tuned_pred)
xgb_tuned_bal_acc = balanced_accuracy_score(y_test, xgb_tuned_pred)

results_summary = pd.DataFrame({
    "model": [
        "Random Forest on 377 engineered features",
        "Random Forest on top 20 before tuning",
        "Random Forest on top 20 after tuning",
        "XGBoost on 377 engineered features",
        "XGBoost on top 20 before tuning",
        "XGBoost on top 20 after tuning"
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, rf_preds_poly),
        rf_reduced_bal_acc,
        rf_tuned_bal_acc,
        balanced_accuracy_score(y_test, xgb_preds_poly),
        xgb_reduced_bal_acc,
        xgb_tuned_bal_acc
    ]
})

print("Best Random Forest parameters:", rf_grid.best_params_)
print("Best XGBoost parameters:", xgb_grid.best_params_)
print()
results_summary


Best Random Forest parameters: {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 200}
Best XGBoost parameters: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 120}



,model,balanced_accuracy
0,Random Forest on 377 engineered features,0.791043
1,Random Forest on top 20 before tuning,0.769356
2,Random Forest on top 20 after tuning,0.806994
3,XGBoost on 377 engineered features,0.836375
4,XGBoost on top 20 before tuning,0.830348
5,XGBoost on top 20 after tuning,0.833134


## Task 3. Combine your models using stacking by implementing a meta learner that uses out-of-fold predictions, and you should compare the performance of your base models with the stacked model.

In [37]:
# OOF stacking using the tuned reduced models

def get_class_1_proba(model, X):
    proba = model.predict_proba(X)
    classes = list(model.classes_)
    if 1 not in classes:
        raise ValueError(f"Class 1 not found in model.classes_: {classes}")
    class_1_index = classes.index(1)
    return proba[:, class_1_index]

meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_oof = np.zeros(len(X_train_rf_20))
xgb_oof = np.zeros(len(X_train_xgb_20))

rf_test_stack = np.zeros((len(X_test_rf_20), meta_skf.n_splits))
xgb_test_stack = np.zeros((len(X_test_xgb_20), meta_skf.n_splits))

for fold, (train_idx, val_idx) in enumerate(meta_skf.split(X_train_rf_20, y_train)):
    rf_fold = clone(rf_best)
    xgb_fold = clone(xgb_best)

    rf_fold.fit(X_train_rf_20.iloc[train_idx], y_train.iloc[train_idx])
    xgb_fold.fit(X_train_xgb_20.iloc[train_idx], y_train.iloc[train_idx])

    rf_oof[val_idx] = get_class_1_proba(rf_fold, X_train_rf_20.iloc[val_idx])
    xgb_oof[val_idx] = get_class_1_proba(xgb_fold, X_train_xgb_20.iloc[val_idx])

    rf_test_stack[:, fold] = get_class_1_proba(rf_fold, X_test_rf_20)
    xgb_test_stack[:, fold] = get_class_1_proba(xgb_fold, X_test_xgb_20)

X_meta_train = np.column_stack([rf_oof, xgb_oof])
X_meta_test = np.column_stack([
    rf_test_stack.mean(axis=1),
    xgb_test_stack.mean(axis=1)
])

meta_model = LogisticRegression(random_state=42)
meta_model.fit(X_meta_train, y_train)

stack_pred = meta_model.predict(X_meta_test)
stack_bal_acc = balanced_accuracy_score(y_test, stack_pred)

comparison_df = pd.DataFrame({
    "model": [
        "Tuned Random Forest (top 20)",
        "Tuned XGBoost (top 20)",
        "Stacked meta learner"
    ],
    "balanced_accuracy": [
        rf_tuned_bal_acc,
        xgb_tuned_bal_acc,
        stack_bal_acc
    ]
})

comparison_df


,model,balanced_accuracy
0,Tuned Random Forest (top 20),0.806994
1,Tuned XGBoost (top 20),0.833134
2,Stacked meta learner,0.797577


## Additional checks

In [38]:

print("Random Forest classification report")
print(classification_report(y_test, rf_tuned_pred))

print("XGBoost classification report")
print(classification_report(y_test, xgb_tuned_pred))

print("Stacked model classification report")
print(classification_report(y_test, stack_pred))


Random Forest classification report
              precision    recall  f1-score   support

           0       0.94      0.77      0.85      7431
           1       0.54      0.84      0.66      2338

    accuracy                           0.79      9769
   macro avg       0.74      0.81      0.75      9769
weighted avg       0.84      0.79      0.80      9769

XGBoost classification report
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      7431
           1       0.58      0.86      0.69      2338

    accuracy                           0.82      9769
   macro avg       0.77      0.83      0.78      9769
weighted avg       0.86      0.82      0.83      9769

Stacked model classification report
              precision    recall  f1-score   support

           0       0.90      0.92      0.91      7431
           1       0.72      0.68      0.70      2338

    accuracy                           0.86      9769
   macro avg       0.81   

## Task 4. In a markdown cell at the end, evaluate your results by describing how performance changed as you reduced features, which features were consistently important, which features you removed and why, whether stacking improved performance, and what you learned about how your model behaves.


### Evaluation

The workflow started with 377 engineered features created from the reduced feature-engine style dataset. I then reduced the feature space to 20 features for each model by combining two ranking methods from the demo notebook: built-in feature importance and permutation importance. This kept the most useful interaction terms while removing many lower-value features.

Performance should be compared in three stages: the full 377-feature models, the reduced 20-feature models before tuning, and the reduced 20-feature models after tuning. In my results table above, the reduced feature sets make it easier to see whether the most useful interactions are carrying most of the predictive signal. If the reduced models stay close to the full models, that suggests much of the larger feature set was redundant.

The features that were consistently important are the ones that appear in both top-20 lists. These often include terms built from age, educational-num, marital-status, relationship, capital-gain, hours-per-week, and occupation-related information. Those variables are also plausible drivers of income in this dataset, so their repeated appearance is a useful model behavior check.

I removed features that ranked poorly by both feature importance and permutation importance. I also removed many engineered interaction terms that did not appear near the top after ranking. The reason for removing them was to reduce complexity, lower runtime, and make the models easier to interpret without keeping weak or redundant terms.

Stacking should be judged by comparing the tuned Random Forest, tuned XGBoost, and the stacked meta learner in the comparison table. If the stacked model performs better, then the two base models are providing complementary information. If it does not improve much, then the base models may already be making similar decisions.

What I learned is that tree-based models can still work well after a large feature reduction if the ranking process keeps the strongest original variables and interactions. I also learned that not all engineered features help equally, and that tuning a smaller feature set is more manageable than tuning the full 377-feature design.
